In [14]:
# %%
# create_vector_db.ipynb
# Cell 1: Imports & Environment Setup

import os
import re
from typing import List

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector
import traceback

# --- Environment Variable & Setup ---
# IMPORTANT: It's recommended to set this as a true environment variable.
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)


In [15]:
# %%
# Cell 2: Configuration

PDF_DIRECTORY_PATH = "./FTP_Chapters/"
CONNECTION_STRING = (__import__('sys').path.insert(0, '..') or __import__('config').DB_CONNECTION_STRING)
COLLECTION_NAME = "VectorDB"

In [16]:
# %%
# Cell 3: PDF Loading and Chunking Function

def load_and_chunk_pdfs(pdf_directory_path: str) -> List[Document]:
    """
    Loads all PDFs from a directory and splits them into semantic chunks.

    This function iterates through all PDF files in the given directory,
    extracts their text, and then splits the text into chunks based on
    section headers (e.g., "1.1 Introduction"). This creates more
    contextually relevant document chunks for the RAG system.

    Args:
        pdf_directory_path: The path to the directory containing PDF files.

    Returns:
        A list of Document objects, where each document represents a
        semantic chunk of a source PDF.
    """
    print(f"--- Loading and chunking all PDFs from directory: '{pdf_directory_path}' ---")
    all_documents = []

    if not os.path.isdir(pdf_directory_path):
        raise FileNotFoundError(f"The specified directory does not exist: {pdf_directory_path}")

    pdf_files = [f for f in os.listdir(pdf_directory_path) if f.lower().endswith(".pdf")]

    if not pdf_files:
        print(f"WARNING: No PDF files found in '{pdf_directory_path}'.")
        return []

    for pdf_file in pdf_files:
        print(f"    -> Processing file: {pdf_file}")
        file_path = os.path.join(pdf_directory_path, pdf_file)
        loader = PyMuPDFLoader(file_path)
        raw_docs = loader.load()
        original_full_text = "".join(doc.page_content for doc in raw_docs)

        # Split the document by section headers (e.g., "1.1 Section Title")
        header_pattern = r"(?m)^\s*(\d{1,2}\.\d{1,2}\s.*?)$"
        split_text = re.split(header_pattern, original_full_text)

        chunks_for_this_file = []
        if split_text[0].strip():
            intro_doc = Document(
                page_content=split_text[0].strip(),
                metadata={"section_header": "Introduction", "source_file": pdf_file}
            )
            chunks_for_this_file.append(intro_doc)

        # Process the main sections
        for i in range(1, len(split_text), 2):
            section_header = split_text[i].strip()
            section_content = split_text[i+1].strip()
            if section_header and section_content:
                combined_content = f"{section_header}\n\n{section_content}"
                section_doc = Document(
                    page_content=combined_content,
                    metadata={"section_header": section_header, "source_file": pdf_file}
                )
                chunks_for_this_file.append(section_doc)

        all_documents.extend(chunks_for_this_file)
        print(f"      -> Split into {len(chunks_for_this_file)} chunks.")

    print(f"\n--- Successfully split {len(pdf_files)} PDF files into {len(all_documents)} total semantic chunks. ---")
    return all_documents

In [17]:
# %%
# Cell 4: Main Execution

try:
    # 1. Load and process the source documents
    document_chunks = load_and_chunk_pdfs(PDF_DIRECTORY_PATH)

    if not document_chunks:
        print("No documents were loaded. Exiting.")
    else:
        # 2. Initialize the embedding model
        print("\n--- Initializing embedding model... ---")
        embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
        print("    -> Embedding model initialized.")

        # 3. Create the vector store and ingest the documents
        print("\n--- Creating/Updating Vector Store in PostgreSQL ---")
        print(f"    -> This will delete and recreate the '{COLLECTION_NAME}' collection.")

        # PGVector.from_documents is the method to use for the initial creation.
        # It handles embedding the documents and inserting them into the database.
        db = PGVector.from_documents(
            documents=document_chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            connection=CONNECTION_STRING,
            pre_delete_collection=True,  # Ensures we start fresh
        )
        print("\n--- SUCCESS: Vector database has been created and populated successfully. ---")
        print("You can now run the 'rag_agent_app.ipynb' notebook to ask questions.")

except Exception as e:
    print(f"\nAN ERROR OCCURRED: {e}")
    traceback.print_exc()

--- Loading and chunking all PDFs from directory: './FTP_Chapters/' ---
    -> Processing file: FTP2023_Chapter07.pdf
      -> Split into 14 chunks.
    -> Processing file: FTP2023_Chapter09.pdf
      -> Split into 14 chunks.
    -> Processing file: FTP2023_Chapter05.pdf
      -> Split into 16 chunks.
    -> Processing file: [UPDATED] CHAPTER 2 OF FTP.pdf
      -> Split into 68 chunks.
    -> Processing file: FTP2023_Chapter08.pdf
      -> Split into 11 chunks.
    -> Processing file: FTP2023_Chapter01.pdf
      -> Split into 36 chunks.
    -> Processing file: FTP2023_Chapter03.pdf
      -> Split into 9 chunks.
    -> Processing file: FTP2023_Chapter11.pdf
      -> Split into 64 chunks.
    -> Processing file: FTP2023_Chapter04 as on 13.02.2025.pdf
      -> Split into 74 chunks.
    -> Processing file: FTP2023_Chapter10.pdf
      -> Split into 13 chunks.
    -> Processing file: Chapter 6 FTP 2023.pdf
      -> Split into 26 chunks.

--- Successfully split 11 PDF files into 345 total sem

In [5]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 52.4 MB/s eta 0:00:00


In [8]:
# debug_pdf_text.py

import os
import re
from langchain_community.document_loaders import PyMuPDFLoader

# --- CONFIGURATION ---
PDF_DIRECTORY_PATH = "./FTP_Chapters/"
PROBLEM_PDF_FILE = "FTP2023_Chapter01.pdf"  # <-- CHANGE THIS TO THE FILE YOU'RE DEBUGGING
OUTPUT_TEXT_FILE = "debug_output.txt"
# ---------------------

file_path = os.path.join(PDF_DIRECTORY_PATH, PROBLEM_PDF_FILE)

print(f"Loading text from: {file_path}")
loader = PyMuPDFLoader(file_path)
raw_docs = loader.load()
full_text = "\n".join(doc.page_content for doc in raw_docs)

print(f"Saving extracted text to: {OUTPUT_TEXT_FILE}")
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write(full_text)

print("\n--- DEBUGGING INFO ---")
header_pattern = r"(?m)^(\d+(?:\.\d+)*\s+.*?)$"
found_headers = re.findall(header_pattern, full_text)

print(f"Found {len(found_headers)} headers in '{PROBLEM_PDF_FILE}':")
for i, header in enumerate(found_headers):
    print(f"  {i+1}. {header}")

print(f"\nProcess complete. Please open '{OUTPUT_TEXT_FILE}' to inspect the full text.")

Loading text from: ./FTP_Chapters/FTP2023_Chapter01.pdf
Saving extracted text to: debug_output.txt

--- DEBUGGING INFO ---
Found 53 headers in 'FTP2023_Chapter01.pdf':
  1. 1
Legal Framework and  
  2. 2
Legal Framework and Trade Facilitation
  3. 3
A.	 LEGAL FRAMEWORK
  4. 1.00	Legal Basis of Foreign Trade Policy
  5. 5 of the Foreign Trade (Development & Regulation) Act, 
  6. 1992 (No. 22 of 1992) [FT (D&R) Act], as amended.
  7. 1.01	Duration of FTP
  8. 1.02	Amendment to FTP
  9. 1.03	Hand Book of Procedures (HBP) and 
  10. 1.04 Specific provision to prevail over the 
  11. 1.05	Transitional Arrangements
  12. 1
Foreign Trade Policy 2023
  13. 4
(b) 	 Item wise Import/Export Policy is delineated in the 
  14. 2.17 of HBP 2023. Bill of Lading and Shipping Bill 
  15. 1.06	National Committee on Trade 
  16. 1.07	DGFT as a facilitator of exports/
  17. 1.08	Free passage of Export Consignment
  18. 1.09	No seizure of export related Stock
  19. 5
1.10	Export of perishable agricultural

In [12]:
# build_vectordb.py
# A complete, robust script to load, intelligently chunk, and embed documents into a PGVector database.

import os
import re
import traceback
from typing import List, Optional

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector

# --- 1. CONFIGURATION ---
# Set these variables to match your environment.

# --- Environment Variable & Setup ---
# IMPORTANT: It's recommended to set this as a true environment variable.
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)

# --- Database & File Paths ---
PDF_DIRECTORY_PATH = "./FTP_Chapters/"
CONNECTION_STRING = (__import__('sys').path.insert(0, '..') or __import__('config').DB_CONNECTION_STRING)
COLLECTION_NAME = "VectorDB" # This will be the name of your table in the database

# --- Chunking Parameters ---
# The minimum number of characters a chunk must have to be considered useful.
MIN_CHUNK_CHAR_LENGTH = 200
# The regex pattern to identify section headers (e.g., "1.1 Section Title").
HEADER_PATTERN = r"(?m)^(\d+(?:\.\d+)*\s+.*?)$"


# --- 2. HELPER & CORE LOGIC FUNCTIONS ---

def _find_chapter_title(text: str) -> Optional[str]:
    """Helper function to find the main chapter title from the text."""
    match = re.search(r"^\s*Chapter\s+\d+\s*\n(.*?)\n", text, re.MULTILINE | re.IGNORECASE)
    return match.group(1).strip() if match else None

def load_and_chunk_pdfs(pdf_directory_path: str) -> List[Document]:
    """
    Loads all PDFs from a directory and applies an intelligent, context-aware chunking strategy.

    This function processes PDFs by:
    1. Splitting them based on section headers.
    2. Intelligently merging small, stray sections with subsequent ones to maintain context.
    3. Filtering out any resulting chunks that are too small to be useful.
    """
    print(f"--- Loading and chunking all PDFs from directory: '{pdf_directory_path}' ---")
    all_final_chunks = []

    if not os.path.isdir(pdf_directory_path):
        raise FileNotFoundError(f"The specified directory does not exist: {pdf_directory_path}")

    pdf_files = [f for f in os.listdir(pdf_directory_path) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print(f"WARNING: No PDF files found in '{pdf_directory_path}'.")
        return []

    for pdf_file in pdf_files:
        print(f"\n==================================================")
        print(f"    -> Processing file: {pdf_file}")
        print(f"==================================================")

        file_path = os.path.join(pdf_directory_path, pdf_file)

        try:
            loader = PyMuPDFLoader(file_path)
            raw_text = "\n".join(doc.page_content for doc in loader.load())

            chapter_title = _find_chapter_title(raw_text) or "Unknown Chapter"
            print(f"      -> Chapter Title: '{chapter_title}'")

            # 1. Initial Split
            # Splits the text into a list: [intro_text, header_1, content_1, header_2, content_2, ...]
            split_text = re.split(HEADER_PATTERN, raw_text)

            # 2. Re-structure into (header, content) pairs
            # The first element is the introduction before any headers
            intro_content = split_text[0].strip()

            # Create pairs of (header, content)
            sections = []
            for i in range(1, len(split_text), 2):
                header = split_text[i].strip()
                content = split_text[i+1].strip() if i + 1 < len(split_text) else ""
                sections.append({"header": header, "content": content})

            # 3. Intelligent Merging Logic
            merged_sections = []
            temp_content = ""

            if intro_content:
                temp_content += intro_content + "\n\n"

            for i, section in enumerate(sections):
                # If the content of a section is too small, it's likely a stray header.
                # We append its header and its small content to the temporary buffer
                # and continue, effectively merging it with the *next* valid section.
                if len(section['content']) < MIN_CHUNK_CHAR_LENGTH and i < len(sections) - 1:
                    print(f"      -> MERGING small chunk '{section['header']}'...")
                    temp_content += f"{section['header']}\n\n{section['content']}\n\n"
                    continue

                # This is a valid section. Combine the buffered content (if any) with this section.
                final_content = (temp_content + f"{section['header']}\n\n{section['content']}").strip()

                # Only create a document if it's substantial
                if len(final_content) >= MIN_CHUNK_CHAR_LENGTH:
                    merged_sections.append(Document(
                        page_content=final_content,
                        metadata={
                            "source_file": pdf_file,
                            "chapter_title": chapter_title,
                            "section_header": section['header']
                        }
                    ))
                else:
                    print(f"      -> SKIPPING final chunk for '{section['header']}' (too small).")

                # Reset the buffer
                temp_content = ""

            print(f"      -> Split into {len(merged_sections)} useful, context-aware chunks.")
            all_final_chunks.extend(merged_sections)

        except Exception as e:
            print(f"      -> ERROR processing file {pdf_file}: {e}")
            traceback.print_exc()

    print(f"\n--- Successfully processed {len(pdf_files)} PDF files into {len(all_final_chunks)} total chunks. ---")
    return all_final_chunks


# --- 3. MAIN EXECUTION BLOCK ---

if __name__ == "__main__":
    try:
        # Step 1: Load and process the source documents using the robust chunking function
        document_chunks = load_and_chunk_pdfs(PDF_DIRECTORY_PATH)

        if not document_chunks:
            print("\nNo documents were processed. Exiting.")
        else:
            # Step 2: Initialize the embedding model
            print("\n--- Initializing embedding model... ---")
            # Using a high-quality Google embedding model
            embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
            print("    -> Embedding model initialized.")

            # Step 3: Create the vector store and ingest the documents
            print("\n--- Creating/Updating Vector Store in PostgreSQL ---")
            print(f"    -> This will DELETE and RECREATE the '{COLLECTION_NAME}' collection.")
            print(f"    -> Embedding and ingesting {len(document_chunks)} documents. This may take a few minutes...")

            # PGVector.from_documents is the method to use for the initial creation.
            # It handles embedding the documents and inserting them into the database.
            # `pre_delete_collection=True` ensures we start with a clean slate.
            db = PGVector.from_documents(
                documents=document_chunks,
                embedding=embeddings,
                collection_name=COLLECTION_NAME,
                connection=CONNECTION_STRING,
                pre_delete_collection=True,
            )

            print("\n\n--- ✅ SUCCESS: Vector database has been created and populated successfully. ---")
            print("You can now run the 'rag_agent_app.ipynb' notebook to ask questions.")

    except Exception as e:
        print(f"\n\n--- ❌ AN ERROR OCCURRED ---")
        print(f"Error: {e}")
        traceback.print_exc()

--- Loading and chunking all PDFs from directory: './FTP_Chapters/' ---

    -> Processing file: FTP2023_Chapter07.pdf
      -> Chapter Title: 'Foreign Trade Policy 2023'
      -> MERGING small chunk '63
Deemed Exports'...
      -> MERGING small chunk '64
Deemed Exports'...
      -> MERGING small chunk '65
7.00	Objective'...
      -> MERGING small chunk '7
Foreign Trade Policy 2023'...
      -> MERGING small chunk '67
7.05	Conditions for refund of Terminal'...
      -> Split into 14 useful, context-aware chunks.

    -> Processing file: FTP2023_Chapter09.pdf
      -> Chapter Title: 'Foreign Trade Policy 2023'
      -> MERGING small chunk '75
Promoting Cross Border Trade in'...
      -> MERGING small chunk '76
Promoting Cross Border Trade in Digital Economy'...
      -> MERGING small chunk '9.03 	E-Commerce Platform'...
      -> MERGING small chunk '9.04 	E-Commerce Export Logistics Provider'...
      -> MERGING small chunk '9
Foreign Trade Policy 2023'...
      -> MERGING small chunk '

In [11]:
!pip install langchain-community langchain-core langchain-google-genai langchain-postgres pymupdf pgvector psycopg2-binary

In [3]:
# build_vectordb_unstructured_v3_corrected.py
# Corrected version to fix the UnboundLocalError.

import os
import traceback
from typing import List

from langchain_unstructured import UnstructuredLoader

from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector

# --- 1. CONFIGURATION ---

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)

PDF_DIRECTORY_PATH = "./FTP_Chapters/"
CONNECTION_STRING = (__import__('sys').path.insert(0, '..') or __import__('config').DB_CONNECTION_STRING)
COLLECTION_NAME = "VectorDB"
MIN_CHUNK_CHAR_LENGTH = 300


# --- 2. THE CHUNKING LOGIC (CORRECTED) ---

def load_and_chunk_pdfs_by_layout(pdf_directory_path: str) -> List[Document]:
    print(f"--- Loading and chunking all PDFs from directory: '{pdf_directory_path}' using Unstructured.io ---")
    all_final_chunks = []

    if not os.path.isdir(pdf_directory_path):
        raise FileNotFoundError(f"The specified directory does not exist: {pdf_directory_path}")

    pdf_files = [f for f in os.listdir(pdf_directory_path) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print(f"WARNING: No PDF files found in '{pdf_directory_path}'.")
        return []

    for pdf_file in pdf_files:
        print(f"\n==================================================")
        print(f"    -> Processing file: {pdf_file}")
        print(f"==================================================")

        # --- THIS IS THE CORRECTED LINE ---
        file_path = os.path.join(pdf_directory_path, pdf_file)

        try:
            loader = UnstructuredLoader(file_path, strategy="hi_res", mode="elements")
            elements = loader.load()

            chunks_for_this_file = []
            current_chunk_content = []
            current_section_header = "Introduction"

            for element in elements:
                is_title = element.metadata.get('category') == "Title"

                if is_title:
                    if current_chunk_content:
                        chunk_text = "\n".join(current_chunk_content)
                        if len(chunk_text) > MIN_CHUNK_CHAR_LENGTH:
                            chunks_for_this_file.append(Document(
                                page_content=f"{current_section_header}\n\n{chunk_text}",
                                metadata={"source_file": pdf_file, "section_header": current_section_header}
                            ))

                    current_section_header = element.page_content
                    current_chunk_content = []
                else:
                    current_chunk_content.append(element.page_content)

            if current_chunk_content:
                chunk_text = "\n".join(current_chunk_content)
                if len(chunk_text) > MIN_CHUNK_CHAR_LENGTH:
                    chunks_for_this_file.append(Document(
                        page_content=f"{current_section_header}\n\n{chunk_text}",
                        metadata={"source_file": pdf_file, "section_header": current_section_header}
                    ))

            print(f"      -> Successfully split into {len(chunks_for_this_file)} semantic chunks.")
            all_final_chunks.extend(chunks_for_this_file)

        except Exception as e:
            print(f"      -> ERROR processing file {pdf_file}: {e}")
            traceback.print_exc()

    print(f"\n--- Successfully processed {len(pdf_files)} PDF files into {len(all_final_chunks)} total chunks. ---")
    return all_final_chunks


# --- 3. MAIN EXECUTION BLOCK ---

if __name__ == "__main__":
    try:
        document_chunks = load_and_chunk_pdfs_by_layout(PDF_DIRECTORY_PATH)

        if not document_chunks:
            print("\nNo documents were processed. Exiting.")
        else:
            print("\n--- Initializing embedding model... ---")
            embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
            print("    -> Embedding model initialized.")

            print("\n--- Creating/Updating Vector Store in PostgreSQL ---")
            print(f"    -> This will DELETE and RECREATE the '{COLLECTION_NAME}' collection.")
            print(f"    -> Embedding and ingesting {len(document_chunks)} documents. This may take a few minutes...")

            db = PGVector.from_documents(
                documents=document_chunks,
                embedding=embeddings,
                collection_name=COLLECTION_NAME,
                connection=CONNECTION_STRING,
                pre_delete_collection=True,
            )

            print("\n\n--- ✅ SUCCESS: Vector database has been created and populated successfully. ---")
            print("You can now run the 'rag_agent_app.ipynb' notebook to ask questions.")

    except Exception as e:
        print(f"\n\n--- ❌ AN ERROR OCCURRED ---")
        print(f"Error: {e}")
        traceback.print_exc()

--- Loading and chunking all PDFs from directory: './FTP_Chapters/' using Unstructured.io ---

    -> Processing file: FTP2023_Chapter07.pdf
      -> Successfully split into 11 semantic chunks.

    -> Processing file: FTP2023_Chapter09.pdf
      -> Successfully split into 6 semantic chunks.

    -> Processing file: FTP2023_Chapter05.pdf
      -> Successfully split into 9 semantic chunks.

    -> Processing file: [UPDATED] CHAPTER 2 OF FTP.pdf
      -> Successfully split into 46 semantic chunks.

    -> Processing file: FTP2023_Chapter08.pdf
      -> Successfully split into 6 semantic chunks.

    -> Processing file: FTP2023_Chapter01.pdf
      -> Successfully split into 21 semantic chunks.

    -> Processing file: FTP2023_Chapter03.pdf
      -> Successfully split into 6 semantic chunks.

    -> Processing file: FTP2023_Chapter11.pdf
      -> Successfully split into 1 semantic chunks.

    -> Processing file: FTP2023_Chapter04 as on 13.02.2025.pdf
      -> Successfully split into 42 se

In [ ]:
# First, install the new, separate langchain-unstructured package
!pip install -U langchain-unstructured

# Then, ensure the core unstructured package has the local models
!pip install "unstructured[local-inference]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 68.5 MB/s eta 0:00:00
  Attempting uninstall: onnxruntime
    Found existing installation: onnxruntime 1.22.0
    Uninstalling onnxruntime-1.22.0:
      Successfully uninstalled onnxruntime-1.22.0


Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
^C


In [4]:
# In a notebook cell, run this:
!apt-get update && sudo apt-get install -y poppler-utils

# Or in your Linux terminal, run this:
!apt-get update
!apt-get install -y poppler-utils

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://packages.cloud.google.com/apt gcsfuse-jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,801 kB]
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,040 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubun